# Reliable and Efficient LLM Outputs with Mellea + Granite OSS Libraries
### ICML 2026 EXPO Workshop

This tutorial will introduce the basics of Generative Computing through a series of labs. Everything you need is in this Notebook.

During this tutorial, we will:
1. Get up an running with Mellea.
2. See the Instruct - Validate - Repair pattern in action.
3. See how you can add your own customization to Mellea.
4. See how Mellea can Make Small Models Great and take you from Demo to Deployment.

### Three IBM open-source tools used in this workshop
- **[Mellea](https://github.com/generative-computing/mellea)** — structured generation, requirements, sampling strategies
- **[Granite Libraries](https://huggingface.co/collections/ibm-granite/granite-libraries)** - adapter functions (answerability, context, relevance, attribution, etc...) via `mellea.stdlib.components.intrinsic` (runs locally via Hugging Face Transformers or with vLLM)
- **[Granite Switch](https://github.com/generative-computing/granite-switch)** — adapter functions packaged as a single model (runs through vLLM in Mellea or directly with Hugging Face Transformers)

## Getting Started

Run the first cell during our introduction. The first cell will:
 * download an install ollama on your Colab instance
 * download the `granite` and `gpt-oss` model weights


In [16]:
# Install ollama. Uncomment if you don't have ollama installed / running.
# !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
# !nohup ollama serve >/dev/null 2>&1 &

# Download the granite4.1:3b weights.
!ollama pull granite4.1:3b

# install Mellea.
!uv pip install "mellea[hf,docling,sandbox,tools]==0.6.0"

# install additional dependencies.
!uv pip install matplotlib "omegaconf>=2.3" "ipywidgets>=8.1.8"

# Get supporting files.
!curl -L https://nfulton.org/atai26.tar.gz | tar -xzf -

# Some UI niceness.
from IPython.display import HTML, display  # noqa: E402
def set_css(*args, **kwargs):
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))
get_ipython().events.register("pre_run_cell", set_css)

# Comment if logging is not verbose enough.
import logging
logging.getLogger("mellea").setLevel(logging.ERROR)



]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 662b0626cd58: 100% ▕██████████████████▏ 2.1 GB                         
pulling 89a0ab46e638: 100% ▕██████████████████▏ 6.8 KB                         
pulling 58d1e17ffe51: 100% ▕██████████████████▏  11 KB                         
pulling 87d22d127f16: 100% ▕██████████████████▏  417 B                         
verifying sha256 digest 
writing manifest 
success 
Using Python 3.12.13 environment at: /Users/jake/Desktop/.venv
Checked 1 package in 89ms
Using Python 3.12.13 environment at: /Users/jake/Desktop/.venv
Checked 3 packages in 16ms
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7445k  100 744

# Module 1: Hello, Mellea!

Running `mellea.start_session()` initialize a new `MelleaSession`. The session holds three things:
1. The model to use for this session. In this tutorial we will use granite4.1:3b.
2. An inference engine; i.e., the code that actually calls our model. We will be using ollama, but you can also use Huggingface or any OpenAI-compatible endpoint.
3. A `Context`, which tells Mellea how to remember context between requests. This is sometimes called the "Message History" in other frameworks. Throughout this tutorial, we will be using a `SimpleContext`. In `SimpleContext`s, **every request starts with a fresh context**. There is no preserved chat history between requests. Mellea provides other types of context, but today we will not be using those features. See the Tutorial for further details.

In [3]:
import mellea

m = mellea.start_session()

answer = m.chat(
    "tell me some fun trivia about IBM and the early history of AI."
)
print(answer.content)

Sure! Here are some interesting tidbits about IBM and its involvement in the early history of artificial intelligence (AI):

1. **1956 Dartmouth Conference**: Often considered the birthplace of AI as a field, the "Dartmouth Summer Research Project on Artificial Intelligence" was funded by a grant from IBM to John McCarthy, Marvin Minsky, Nathaniel Rochester, and Claude Shannon at Dartmouth College in 1956. The project aimed to explore how machines could be made to solve problems currently solved by humans.

2. **IBM's PDP-8**: In the early 1960s, IBM introduced the PDP-8 (Programmed Data Processor-8), one of the first minicomputers that became widely available for research and commercial use. This machine played a crucial role in making AI accessible to more researchers and businesses due to its affordability compared to mainframe computers.

3. **Geoffrey Hinton at IBM**: Renowned AI researcher Geoffrey E. Hinton spent several years (1970s-1980s) working at the University of Toronto, 

In [4]:
PAPER_ABSTRACT = """
Title: LLMON: An LLM-native Markup Language to Leverage Structure and Semantics at the LLM Interface
Authors: Michael Hind, Basel Shbita, Bo Wu, Farhan Ahmed, Chad DeLuca, Nathan Fulton, David Cox, Dan Gutfreund
Abstract: Textual Large Language Models (LLMs) provide a simple and familiar interface: a string of text is used for both input and output. However, the information conveyed to an LLM often has a richer structure and semantics, which is not conveyed in a string. For example, most prompts contain both instructions (“Summarize this paper into a paragraph”) and data (the paper to summarize), but these are usually not distinguished when passed to the model. This can lead to model confusion and security risks, such as prompt injection attacks. This work addresses this shortcoming by introducing an LLM-native mark-up language, LLMON (LLM Object Notation, pronounced “Lemon”), that enables the structure and semantic metadata of the text to be communicated in a natural way to an LLM. This information can then be used during model training, model prompting, and inference implementation, leading to improvements in model accuracy, safety, and security. This is analogous to how programming language types can be used for many purposes, such as static checking, code generation, dynamic checking, and IDE highlighting. We discuss the general design requirements of an LLM-native markup language, introduce the LLMON markup language and show how it meets these design requirements, describe how the information contained in a LLMON artifact can benefit model training and inference implementation, and provide some preliminary empirical evidence of its value for both of these use cases. We also discuss broader issues and research opportunities that are enabled with an LLM-native approach.
"""

result = m.instruct(
    "Answer this question about the research paper abstract below.\n\n"
    "Abstract:\n{{abstract}}\n\nQuestion: {{question}}",
    user_variables={
        "abstract": PAPER_ABSTRACT,
        "question": "How does the paper propose increasing model accuracy and safety?",
    },
)

print(str(result))

The paper proposes increasing model accuracy and safety by introducing an LLM‑native markup language called LLMON (LLM Object Notation, pronounced “Lemon”). This markup language allows the structure and semantic metadata of text—such as distinguishing between instructions and data in a prompt—to be communicated naturally to the Large Language Model (LLM). By doing so:

1. **Improved Accuracy:** The explicit separation of information types helps the model better understand the intended context and requirements, reducing ambiguity that can lead to incorrect or irrelevant outputs.

2. **Enhanced Safety:** Properly distinguishing between instructions and data mitigates security risks like prompt injection attacks, where malicious input could trick the model into performing unintended actions. This structured communication ensures that the model adheres more strictly to the intended behavior defined by the user.

Overall, LLMON facilitates better integration of semantic metadata during both

In [5]:
from mellea.backends.model_options import ModelOption

session = mellea.start_session(
    context_type='chat',
)

model_opts = { 
    ModelOption.SYSTEM_PROMPT: (
        f"You are a research assistant. Answer questions using only this abstract:\n"
        f"{PAPER_ABSTRACT}"
    ),
    ModelOption.MAX_NEW_TOKENS: 30,
    ModelOption.SEED: 1
}

turn1 = session.chat("What information do most prompts contain?", model_options=model_opts)
print(f"Turn 1: {turn1.content}\n")

turn2 = session.chat("Why does that cause model confusion?", model_options=model_opts)
print(f"Turn 2: {turn2.content}\n")

Turn 1: Most prompts typically contain a combination of instructions (e.g., "Summarize this paper into a paragraph") and data relevant to the task (e

Turn 2: The blending of instructions and data in a single string can lead to model confusion because the Large Language Model (LLM) may not be able to distinguish



In [7]:
from mellea.stdlib import functional as mfuncs
from mellea.stdlib.context import ChatContext
from mellea.stdlib.components import Instruction

async def functional_example():
    backend = session.backend
    ctx = ChatContext()

    instruction = Instruction(f"Summarize the abstract: {PAPER_ABSTRACT}")

    # Could also use mfuncs.act() for synchronous interface.
    out, ctx = await mfuncs.aact(instruction, ctx, backend)

    print(out.value)

# asyncio.run(functional_example())
await functional_example()

The abstract describes a new markup language called LLMON (LLM Object Notation) designed to enhance the interaction between Textual Large Language Models (LLMs) and their users by conveying richer structure and semantic information. Traditional LLM interfaces use plain text for both input and output, which can lead to confusion and security issues like prompt injection attacks because instructions and data are not clearly distinguished. LLMON addresses this by allowing structured metadata to be included in prompts, similar to how programming languages use types for various purposes such as static checking and code generation. The authors outline the design requirements for an LLM-native markup language, introduce LLMON, demonstrate its benefits for model training and inference, and explore broader research opportunities enabled by adopting an LLM-native approach.


In [8]:
from mellea import generative
from pydantic import BaseModel, Field

class PaperAnswer(BaseModel):
    answer: str = Field(description="Direct answer to the question")
    confidence: str = Field(description="One of: high, medium, low")
    relevant_quote: str = Field(
        description="The exact sentence from the abstract that supports the answer, "
                    "or 'Not found in abstract' if none exists"
    )

@generative
def answer_paper_question(abstract: str, question: str) -> PaperAnswer:
    """Answer the question using only information from the abstract.
    If the abstract does not contain enough information, set confidence to 'low'."""
    ...

backend = session.backend
ctx = ChatContext()

result, new_ctx = answer_paper_question(
    backend=backend,
    context=ctx,
    abstract=PAPER_ABSTRACT,
    question="How many tasks were used to evaluate LLMON?",
)

print(f"Answer:     {result.answer}")
print(f"Confidence: {result.confidence}")
print(f"Quote:      {result.relevant_quote}")

Answer:     The abstract does not specify the number of tasks used to evaluate LLMON.
Confidence: low
Quote:      


# Module 2: Instruct-Validate-Repair

Instruct-Validate-Repair is a design pattern for building robust automation using LLMs. The idea is simple:
1. Instruct the model to perform a task and specify requirements on the output of the task.
2. Validate that these requirements are satisfied by the model's output.
3. If any requirements fail, try to repair.

| Type | Class | Validated by | Cost |
|------|-------|-------------|------|
| **Programmatic** | `req(..., validation_fn=simple_validate(fn))` | A `str → bool` function you write | Free / cost of Python |
| **LLM-as-judge** | `LLMaJRequirement(description)` | The model itself, given the description | One extra inference call |
| **Granite adapter functions** | `check_answerability`, `check_context_relevance` (from `mellea.stdlib.components.intrinsic`) | Hugging Face transformers backend | ~5-10% latency overhead (requires `mellea[hf]`) |

In [9]:
PAPER_EXCERPT = """Backbone models We evaluate two backbone models representing different architectural families
and training lineages: Granite-4.0-Micro-Base (Granite Team, 2024) (3B parameters) and Qwen2.5-
3B (Yang et al., 2025). These models serve as representative mid-scale LLMs with competitive
baseline performance. For each backbone, we report results for (i) the base model without post-
training, (ii) conventional post-training using chat-template instruction data, and (iii) post-training
with LLMON-structured corpora under both full fine-tuning and LoRA adaptation.
Training configuration and scaleWe use the structure-aware corpora described in Section 5.2,
comprising (i) LLMON-wrapped public post-training and instruction-tuning data, and (ii) distractor-
infused LLMON variants designed to train identifier-based focus under multi-instruction conditions.
For the baseline post-training runs, we use the same underlying instruction-tuning datasets serialized
using conventional chat-template formatting. Across large-scale public single-turnSFTdata, Alpaca,
and Dolly, the resulting mixture contains roughly3.4 million structured examplestotaling2.9
billion tokens. To study scaling effects, we train models on progressive subsets of this corpus
(241M, 436M, 616M tokens) as well as the full 2.9B-token mixture. Post-training is performed using
standardSFTunder both full fine-tuning and parameter-efficient LoRA adaptation. No architectural
modifications are introduced.
Evaluation protocolWe introduce a dedicatedDistractorbenchmark: 100 manually curated
Alpaca instances in multi-instruction form with one instruction selected via “ exec”. An LLM-
as-a-Judge (LLMaJ) scores each response, measuring identifier-based binding and robustness to
distractors. The final score is the average across all instances. We measure robustness and execution
control (RQ1–RQ2) using the Distractor benchmark, which evaes the
└─────┴─────────────────────────────────────────────────┴────────────────────────────────┴──────────────┴──────────────────────────────────────┘

So the answer to your earlier question ("what would a smallmpirically confirmed: not the isolated memorable facts, butquestions that force it to keep a specific attribute bound to a specific entity across a multi-row table. Those produce confident, well-formatted, wrong numbers every time.

The harness is at /tmp/ask.py (takes start end line args + questions on stdin, seed pinned to 1). Want me to run a larger automated sweep to quantify the hallucination rate on table-binding questions vs. single-fact questions, or point it at other sections?

✻ Cogitated for 1m 25s
to the model (system framing + delimited excerpt + question), and the model's response.

Here's the concise index of what's shown, with the ground truth for each:

┌─────┬─────────────────────────┬──────────────────────────────────────┬──────────────────────┬───────────────────────────────────────────────┐
│  #  │ Excerpt (extracted-text │               Question               │     Model output     │                Correct answer                 │
│     │          lines)         │                                      │                      │                                               │
├─────┼─────────────────────────┼──────────────────────────────────────┼──────────────────────┼───────────────────────────────────────────────┤
│ 1   │ 744–831 (Section 6.1,   │ "Which SFT datasets were used?"      │ Alpaca, Dolly, MMLU, │ Alpaca, Dolly (MMLU/GSM8K/IFEval are eval     │
│     │ Evaluation)             │                                      │  GSM8K, IFEval       │ benchmarks, not SFT data)                     │
├─────┼─────────────────────────┼──────────────────────────────────────┼──────────────────────┼───────────────────────────────────────────────┤
│ 2   │ 850–892 (Section 6.2,   │ Granite-3.3-8B-Instruct              │ 43.2 → 72.4          │ 41.6 → 72.0 (43.2 is Qwen's baseline; 72.4 is │
│     │ Inference)              │ baseline→masked Distractor?          │                      │  Granite-4.0-Micro's result)                  │
├─────┼─────────────────────────┼──────────────────────────────────────┼──────────────────────┼───────────────────────────────────────────────┤
│ 3   │ 781–831 (Table 4)       │ Granite-4.0-Micro-Base, LoRA, 616M,  │ 86.80                │ 74.20 (86.80 is Qwen full-fine-tuning at      │
│     │                         │ Distractor?                          │                      │ 616M)                                         │
└─────┴─────────────────────────┴──────────────────────────────────────┴──────────────────────┴───────────────────────────────────────────────┘

A few notes on exact reproducibility:

- Prompt template is the build_prompt() string shown at the top of each block: a fixed instruction line, then --- EXCERPT --- / --- END EXCERPT --- fences around the raw text, then Question: / Answer:.
- Excerpt text is the raw output of pypdf's extract_text() on the arXiv PDF — note it contains the extraction artifacts you can see above (e.g. Backbone modelsWe, standardSFTunder, spaced hyphens in Hendrycks et al .,). Those artifacts are part of the exact input; a cleaned excerpt could shift behavior.
elsWe evaluate two backbone models representing different architectural families
and training lineages: Granite-4.0-Micro-Base (Granite Team, 2024) (3B parameters) and Qwen2.5-
3B (Yang et al., 2025). These models serve as representative mid-scale LLMs with competitive
baseline performance. For each backbone, we report results for (i) the base model without post-
training, (ii) conventional post-training using chat-template instruction data, and (iii) post-training
with LLMON-structured corpora under both full fine-tuning and LoRA adaptation.
Training configuration and scaleWe use the structure-aware corpora described in Section 5.2,
comprising (i) LLMON-wrapped public post-training and instruction-tuning data, and (ii) distractor-
infused LLMON variants designed to train identifier-based focus under multi-instruction conditions.
For the baseline post-training runs, we use the same underlying instruction-tuning datasets serialized
using conventional chat-template formatting. Across large-scale public single-turnSFTdata, Alpaca,
and Dolly, the resulting mixture contains roughly3.4 million structured examplestotaling2.9
billion tokens. To study scaling effects, we train models on progressive subsets of this corpus
(241M, 436M, 616M tokens) as well as the full 2.9B-token mixture. Post-training is performed using
standardSFTunder both full fine-tuning and parameter-efficient LoRA adaptation. No architectural
modifications are introduced.
Evaluation protocolWe introduce a dedicatedDistractorbenchmark: 100 manually curated
Alpaca instances in multi-instruction form with one instruction selected via “ exec”. An LLM-
as-a-Judge (LLMaJ) scores each response, measuring identifier-based binding and robustness to
distractors. The final score is the average across all instances. We measure robustness and execution
control (RQ1–RQ2) using the Distractor benchmark, which evaluates whether a model executes the
instruction explicitly referenced by the “exec” in multi-instruction contexts. Performance reflects
correct identifier-based instruction binding. General capability retention (RQ3) is evaluated using
standard single-instruction benchmarks without distractors, including MMLU (Hendrycks et al .,
2021), GSM8K (Cobbe et al., 2021), and IFEval (Zhou et al.,
ResultsTable 4 compares base models, conventional post-training baselines using chat-template
instruction data, and LLMON-structured post-training. The Distractor column demonstrates that
structured post-training with LLMON substantially improves execution control for both backbones.
Both base models and conventionally post-trained chat-templd) exhibit
near-zero (0.00 and 0.40) accuracy on the Distractor benchmark, indicating weak identifier-based
binding and frequent execution of positionally salient or distractor instructions. After LLMON-
structured fine-tuning, execution accuracy rises dramaticalscales,
15
confirming that explicit instruction-data separation and “exec”-based binding can be reliably learned.
In contrast, models post-trained using conventional chat-template formatting show little improvement
and often degrade performance on the Distractor benchmark, indicating that the gains arise from
structured supervision rather than from additional post-training alone.
Table 4: Comparison of base models, conventional post-traininstruction
 indicate the amount of post-training data used.
Model Tokens MMLU GSM8K IFEval Distractor
Base (no post-training)
Granite-4.0-Micro-Base – 61.52 0.00 38.78 4.40
Qwen-2.5-3B – 65.09 0.15 27.13 27.80
Full fine-tuning baseline (post-training with chat-template data)
Granite-4.0-Micro-Base 2.9B 39.95 0.61 72.07 0.00
Qwen-2.5-3B 2.9B 48.51 1.13 74.05 0.40
Full fine-tuning (post-training with LLMON-structured data)
Granite-4.0-Micro-Base
241M 36.69 26.31 54.75 87.80
436M 36.38 13.12 57.95 87.40
616M 41.95 15.84 65.82 85.80
2.9B 43.35 20.39 68.14 84.00
Qwen-2.5-3B
241M 48.35 23.65 58.86 88.00
436M 48.01 24.18 62.40 87.60
616M 47.92 12.21 65.38 86.80
2.9B 47.26 16.83 71.96 83.40
LoRA baseline (post-training with chat-template data)
Granite-4.0-Micro-Base 2.9B 58.53 7.43 67.08 19.40
Qwen-2.5-3B 2.9B 63.55 14.93 47.26 2.40
LoRA (post-training with LLMON-structured data)
Granite-4.0-Micro-Base
241M 53.72 34.87 54.07 74.40
436M 51.32 20.24 50.58 75.60
616M 56.11 7.05 56.22 74.20
2.9B 54.97 15.77 60.35 75.00
Qwen-2.5-3B
241M 49.57 4.47 33.07 72.20
436M 62.63 7.43 38.80 72.40
616M 48.31 24.03 44.57 69.80
2.9B 55.62 20.24 48.20 70.80
On fully fine-tuned models, scaling effects show that increasing token budgets generally stabilizes
structured behavior and improves performance on instruction-following benchmarks such as IFEval,
though gains are not strictly monotonic across all tasks. Sd supervision
does not catastrophically degrade general capability: while some benchmarks fluctuate relative to
base performance, models retain competitive accuracy on MMLU, GSM8K, and IFEval.
LoRA adaptation also yields large improvements in LLMON-structured execution relative to base
models (typically around 70% compared to near-zero baseline performance), demonstrating that
structural behavior can be acquired in a parameter-efficient regime. However, LoRA consistently
underperforms full fine-tuning on the Distractor benchmark, particularly at larger token scales,
suggesting that deeper weight updates better internalize execution semantics. Interestingly, LoRA
sometimes achieves stronger performance on knowledge-oriented benchmarks such as MMLU,
whereas IFEval tends to benefit more consistently from full mixed
results across models. Across both backbones, full fine-tun highest
Distractor benchmark scores, reinforcing the central claim that LLMON-structured supervision
improves control and multi-instruction robustness rather than merely fitting surface patterns."""

# Initial Failure Mode
session.reset()
hallucinated = session.instruct(
    f"{PAPER_EXCERPT} \n\nWhich SFT datasets were used in training? Output only a list.\n\n",
    model_options={ModelOption.SEED: 1}
)
print("=== Model output ===")
print(str(hallucinated))
print()

=== Model output ===
Alpaca, Dolly, MMLU, GSM8K, IFEval



In [10]:
from typing import List

class ListOutput(BaseModel):
    answer: List[str] = Field(description="Direct answer to the question")

# Fixed
session.reset()
corrected = session.instruct(
    f"{PAPER_EXCERPT} \n\nWhich SFT datasets were used for training?",
    model_options={ModelOption.SEED: 1},
    requirements=[
        "Don't output benchmark datasets."
    ],
    format=ListOutput
)

print(corrected)

{ "answer": ["Alpaca", "Dolly"] }


In [14]:
from mellea.backends.huggingface import LocalHFBackend
from mellea.backends.model_ids import IBM_GRANITE_4_MICRO_3B

hf_backend = LocalHFBackend(model_id=IBM_GRANITE_4_MICRO_3B)

#
from mellea.stdlib.components.intrinsic import rag
citations = rag.find_citations(response=None, documents=[PAPER_ABSTRACT, PAPER_EXCERPT], context=session.ctx, backend=hf_backend) # type: ignore
print(citations[0])

# Or, use the functional approach:
# out, new_ctx = mfuncs.act(
#     Intrinsic(
#         "citations",
#     ),
#     ctx,
#     backend,
# )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

{'response_begin': 0, 'response_end': 33, 'response_text': '{ "answer": ["Alpaca", "Dolly"] }', 'citation_doc_id': None, 'citation_begin': 855, 'citation_end': 1165, 'citation_text': 'For the baseline post-training runs, we use the same underlying instruction-tuning datasets serialized\nusing conventional chat-template formatting. Across large-scale public single-turnSFTdata, Alpaca,\nand Dolly, the resulting mixture contains roughly3.4 million structured examplestotaling2.9\nbillion tokens. '}


# Module 3: Customization

In [15]:
from mellea.backends.adapters.adapter import CustomIntrinsicAdapter
from mellea.stdlib.requirements import ALoraRequirement

class StemboltAdapter(CustomIntrinsicAdapter):
    def __init__(self):
        super().__init__(
            model_id="nfulton/stembolts",
            intrinsic_name="stembolts",
            base_model_name="granite-3.3-2b-instruct",
        )


granite_33_2b_stembolt_adapter = StemboltAdapter()

backend = LocalHFBackend(
    model_id="ibm-granite/granite-3.3-2b-instruct",
)

m = mellea.MelleaSession(backend=backend, ctx=ChatContext())

backend.add_adapter(granite_33_2b_stembolt_adapter)

failure_check = ALoraRequirement(
    "The diagnostic confidence should be in the unit interval and greater than 0.9.",
    intrinsic_name=granite_33_2b_stembolt_adapter.intrinsic_name,
)
failure_check.check_only = True

res = m.instruct(
    "Oil seepage around piston rings suggests seal degradation",
    requirements=[failure_check],
    strategy=None,
)

print("==== Generation =====")
print(f"Model Output: {res}")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/jake/Desktop/.venv/lib/python3.12/site-packages/torch/nn/parameter.py:21: RuntimeWarning: coroutine 'functional_example' was never awaited
  def __instancecheck__(self, instance) -> bool:


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


==== Generation =====
Model Output: Oil seepage around piston rings is a clear indication of potential seal de


# Module 4: Making Small Models Rock: Construction BOM Estimate with Chart Construction

Mellea lets you do Big Model Things with Small Models. With a bit of effort, you can do with a 1B paramter on your laptop what would otherwise require GPT 5.2 with Extended Thinking.

The Open Weight Ecosystem produces really amazing small models. Thereofre, using Small MOdels (plus Mellea) to do Bith Model Things has three major advantages:
1. Low and Predictable Margins
2. Data Sovreignty
3. Local or Vendor Agnostic Inference

Doing Big Models Things with Small Models requires taking a few steps that Mellea is engineered to support:

 * **Decompose the Problem**. Example: even GPT 4.1 would beneift significantly from separating data extraction from chart construction. Breaking 
 * **Externalize Control Flow**. Small models can do a few things at time, but struggle with following lots of instructions in a single large prompt. The good news is that programming has never been more automated. Using claude inter alia to 
 * **Modularize Model Capabilities**. It's really hard to make a _general purpose_ small model that is as good as a large foundation model. However, it is much easier to make small models that are as good as large models at _specific, but still quite general, tasks_. Mixing these capabilities is hard, but that's something we can do programmatically!

In this case study, we'll build a workflow that produces a cost estimate from a bill of materials for a construction project.

Inputs:
 * PDFs containing construction plans
 * PDFs containing product catalogs

The Plan:
1. Load all documents using Docling
2. Extract the BOM from the construction plans
3. Loop over all of the items and find the relevant catalog document, extract the price, and assign category
4. Generate pie chart showing cost breakdown

### Step 1: Extracting a Bill of Materials from Construction Documents

We'll start by parsing the tables in the construction plan to build a bill of materials.

In [ ]:
import logging
logging.getLogger("mellea").setLevel(logging.ERROR)

from mellea.stdlib.components.docs.richdocument import RichDocument
construction_plans = RichDocument.from_document_file("construction_docs/construction_plans.pdf")
print(construction_plans.get_tables()[0].to_markdown())
print(construction_plans.get_tables()[1].to_markdown())

[INFO] 2026-07-06 09:44:46,296 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-07-06 09:44:46,300 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-06 09:44:46,308 [RapidOCR] download_file.py:60: File exists and is valid: /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-06 09:44:46,309 [RapidOCR] main.py:50: Using /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-06 09:44:46,408 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-07-06 09:44:46,408 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-06 09:44:46,410 [RapidOCR] download_file.py:60: File exists and is valid: /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-06 09:44:46,410 [RapidOCR] main.py:50: Using /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_m

| Item                             | Qty         | Notes                                  |
|----------------------------------|-------------|----------------------------------------|
| 2x6x92-5/8 studs                 | 130         | Exterior wall studs                    |
| 2x6 plates, total                | 420 lf      | Double top plate + single bottom plate |
| 2x6 PT sill/bottom plate         | 140 lf      | If bearing directly on concrete        |
| 2x10 header stock                | 96 lf       | For windows and doors                  |
| 1/2" plywood header spacers      | 2 sheets    | For built-up headers                   |
| 2x4 blocking / bracing           | 20 pcs @10' | Blocking, bracing, backing             |
| Premanufactured 30' roof trusses | 21          | 24" o.c. over 40' length               |
| 2x4 gable / roof bracing stock   | 12 pcs @10' | Temporary and permanent bracing        |
| 1x8 fascia board or equivalent   | 160 lf      | Eaves + rakes                

We need to extract only the tables that are part of the bill of materials, and put those tables into a computer-readable format.

In [3]:
from typing import Literal
from mellea.core import ModelOutputThunk
from mellea.stdlib.components.docs.richdocument import Table
from mellea.stdlib.requirements import req, simple_validate
import pydantic

m = mellea.start_session()

class BOMEntry(pydantic.BaseModel):
    item: str
    quantity: int | str
    notes: str
    category: Literal["lumber", "windows", "doors", "other"]

class BOM(pydantic.BaseModel):
    items: list[BOMEntry]

def _bom_entry_is_well_formed(entry: BOMEntry) -> bool:
    """Checks that the BOMEntry quantity is either an integer or 'allowance'."""
    try:
        int(entry.quantity)
        return True
    except ValueError as e:
        if entry.quantity.lower() == "allowance":
            return True
    return False

def _bom_entries_are_well_formed(s: str) -> bool:
    try:
        bom = BOM.model_validate_json(s)
        return all([_bom_entry_is_well_formed(entry) for entry in bom.items])
    except pydantic.ValidationError as e:
        print(f"Failed on table: {s}")
        return False

# Filter out tables that are not lists of construction items.
@mellea.generative
def is_material_list(table_markdown: str) -> Literal["yes", "no"]:
    """Determines if the table contains a list of construction items. Yes means it contains construction materials."""

async def extract_bom(doc: RichDocument):
    bom_routines = list()
    # Fire off async requests for each table.
    for table in doc.get_tables():
        if is_material_list(m, table_markdown=table.to_markdown()) == "yes":
            print("Reformatting table.")
            next_sub_bom = m.ainstruct(
                "Reformat this table to have four columns: item, quantity, type, and notes (optional).",
                grounding_context={'table': table.to_markdown()},
                requirements=[
                    req(
                        "Quantity row should only contain an integer or Allowance",
                        validation_fn=simple_validate(_bom_entries_are_well_formed)
                    ),
                    req(
                        "type should be one of: lumber, windows, doors, other",
                        validation_fn=simple_validate(lambda x: True)
                    ), # note: this is enforced by the Literal type so no check is required.
                ],
                format=BOM
            )
            bom_routines.append(next_sub_bom)
    
    # wait for all of the async work to finish, then concatenate the results.
    bom_thunks: list[ModelOutputThunk] = [await bom_routine for bom_routine in bom_routines]
    boms = [BOM.model_validate_json(await bom_thunk.avalue()) for bom_thunk in bom_thunks]
    
    # Concatente all of the indiviual BOMs into one large list.
    all_items = []
    for bom in boms:
        all_items.extend(bom.items)
    full_bom = BOM(items=all_items)
    return full_bom

bom = await extract_bom(doc=construction_plans)

=== 09:45:00-INFO ======
Starting Mellea session: backend=ollama, model=granite4.1:3b, context=SimpleContext
Reformatting table.
Reformatting table.


  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:45:10-INFO ======
SUCCESS


  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:45:14-INFO ======
SUCCESS


  0%|          | 0/2 [00:03<?, ?it/s]


Let's take a look at some of our extracted items:

In [4]:
# print out some random selection of items as a markdown table.
import random
random_entries = random.choices(bom.items, k=5)
items = {f"item_{i}": entry.model_dump_json() for i, entry in enumerate(random_entries)}
print(m.instruct("Format these items as as a markdown table.", grounding_context=items).value)

  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:45:19-INFO ======
SUCCESS


  0%|          | 0/2 [00:01<?, ?it/s]

| Item                           | Quantity | Notes                                      | Category |
|--------------------------------|----------|--------------------------------------------|----------|
| 2x4 blocking / bracing         | 20       | Blocking, bracing, backing                 | Lumber   |
| Premanufactured 30' roof trusses | 21      | 24\" o.c. over 40' length                  | Other    |
| 2x10 header stock              | 96       | For windows and doors                      | Lumber   |
| 1x8 fascia board or equivalent | 160      | Eaves + rakes                               | Lumber   |
| 1x8 fascia board or equivalent | 160      | Eaves + rakes                               | Lumber   |


We now have our entire bill of materials in a structured format. Let's move on to extracting pricing estimates!

### Step 2: Find Prices for Materials

LLMs make errors confidently. If we ask a model to generate prices for our bill of materials, the model will either hallucinate or execute a web search. The web search path may appear to be a panacea, but in practice runs into many issues -- especially in B2B settings.

A better solution is to ensure that answers are grounded in context. We can do this by leveraging the Granite RAG Intrinsics library, which are special-build model adapters for **calibrated** context relevance and answerability checking.

Let's assume that our builder has negotiated rates from lumber, windows, and doors -- they want to use those rates for these items. We will demonstrate this basic concept by adding pricing to three components of the procurement list.

To continue from here, we'll need to start a new session using the **huggingface** backend -- adapters are a feature that are not yet supported by the ollama backend.

In [5]:
import mellea
from mellea.backends.model_ids import IBM_GRANITE_4_MICRO_3B
from mellea.backends.huggingface import LocalHFBackend
from mellea.stdlib.context import ChatContext

m_hf = mellea.MelleaSession(
    backend=LocalHFBackend(model_id=IBM_GRANITE_4_MICRO_3B)
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Before diving into the bom->price problem, let's start with a simple example of intrinsics. We'll use context relevance and answerability adapter functions to check the relevance of the door and window catalogs for various items in our bill of materials.

In [6]:
from mellea.stdlib.components.docs import Document
from mellea.stdlib.components.docs.richdocument import RichDocument

# Load the windows and doors catalogs, and convert them to plain text (markdown) documents.
rd_doors = RichDocument.from_document_file("construction_docs/product_catalogs/door_product_catalog.pdf")
doors_doc = Document(text=rd_doors.to_markdown())

rd_windows = RichDocument.from_document_file("construction_docs/product_catalogs/north_ridge_windows.docx")
windows_doc = Document(text=rd_windows.to_markdown())

rd_lumber = RichDocument.from_document_file("construction_docs/product_catalogs/cone_mountain_lumber_catalog.xlsx")
lumber_doc = Document(text=rd_lumber.to_markdown())

[INFO] 2026-07-06 09:45:30,134 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-07-06 09:45:30,135 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-06 09:45:30,143 [RapidOCR] download_file.py:60: File exists and is valid: /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-06 09:45:30,143 [RapidOCR] main.py:50: Using /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-06 09:45:30,238 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-07-06 09:45:30,238 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-06 09:45:30,240 [RapidOCR] download_file.py:60: File exists and is valid: /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-06 09:45:30,240 [RapidOCR] main.py:50: Using /Users/jake/Desktop/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_m

Now let's do some context relevance and answerability checks:

In [7]:
from mellea.stdlib.components.intrinsic.rag import check_context_relevance, check_answerability

door_catalog_context_relevance_score = check_context_relevance(
    question="What is the price of the 36x80 Aurora half-moon entry door?",
    document=doors_doc,
    context=ChatContext(),
    backend=m_hf.backend # type: ignore
)
print(f"Door catalog context relevance score for `What is the price of the 36x80 Aurora half-moon entry door?`: {door_catalog_context_relevance_score}")


door_catalog_answerability_score = check_answerability(
    question="What is the price of the 36x80 Aurora half-moon entry door?",
    documents=[doors_doc],
    context=ChatContext(),
    backend=m_hf.backend # type: ignore
)
print(f"Door catalog answerability score for `What is the price of the 36x80 Aurora half-moon entry door?`: {door_catalog_answerability_score}")

=== 09:45:56-INFO ======
Tools for call: dict_keys([])
Door catalog context relevance score for `What is the price of the 36x80 Aurora half-moon entry door?`: relevant
=== 09:46:00-INFO ======
Tools for call: dict_keys([])


/Users/jake/Desktop/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Door catalog answerability score for `What is the price of the 36x80 Aurora half-moon entry door?`: answerable


Let's see what happens if we leave the question the same but substitute the doors catalog for the lumber catalog:

In [8]:
lumber_catalog_context_relevance_score = check_context_relevance(
    question="What is the price of the 36x80 Aurora half-moon entry door?",
    document=lumber_doc,
    context=ChatContext(),
    backend=m_hf.backend # type: ignore
)
print(f"Lumber catalog context_relevance score for `What is the price of the 36x80 Aurora half-moon entry door?`: {lumber_catalog_context_relevance_score}")


lumber_catalog_context_relevance_score = check_answerability(
    question="What is the price of of the 36x80 Aurora half-moon entry door?",
    documents=[lumber_doc],
    context=ChatContext(),
    backend=m_hf.backend # type: ignore
)
print(f"Lumber catalog answerability score for `What is the price of the 36x80 Aurora half-moon entry door?`: {lumber_catalog_context_relevance_score}")

=== 09:46:32-INFO ======
Tools for call: dict_keys([])
Lumber catalog context_relevance score for `What is the price of the 36x80 Aurora half-moon entry door?`: partially relevant
=== 09:46:38-INFO ======
Tools for call: dict_keys([])
Lumber catalog answerability score for `What is the price of the 36x80 Aurora half-moon entry door?`: unanswerable


To have extra confidence, or if we need to refer to the source material directly, we can pair these adapter functions with the  `find_citations` adapter function, which identifies the specific segments of the document were used to answer the question:

In [9]:
from mellea.stdlib.components.chat import Message
from mellea.stdlib.components.intrinsic.rag import find_citations
import json # for pretty-printing the citation results.

price_response = m_hf.instruct("What is the price of a half-moon entry door?", grounding_context={"doors": doors_doc.text})

# Because we are using a SimpleContext, we will need to construct a chat history containing just the most recent question and answer.
ctx = ChatContext()
ctx = ctx.add(Message(role="user", content="What is the price of a half-moon entry door?"))
ctx = ctx.add(Message(role="assistant", content=price_response.value))

citations = find_citations(
    response=price_response.value,
    documents=[doors_doc],
    context=ctx,
    backend=m_hf.backend # type: ignore
)

print(json.dumps(citations, indent=4))

  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:46:51-INFO ======
SUCCESS


  0%|          | 0/2 [00:03<?, ?it/s]


=== 09:46:51-INFO ======
Tools for call: dict_keys([])
[
    {
        "response_begin": 0,
        "response_end": 85,
        "response_text": "The price of a half-moon entry door, specifically the Aurora Half-Moon Entry Door, is",
        "citation_doc_id": null,
        "citation_begin": 459,
        "citation_end": 732,
        "citation_text": "<!-- image -->\n\n## EXTERIOR\n\nEXT-101\n\n## Aurora Half-Moon Entry Door $689 unit price\n\nSIZE\n\n36 x 80 in\n\nCONSTRUCTION\n\nInsulated steel\n\nAPPLICATION\n\nExterior\n\n## Description\n\nA durable insulated entry door with a classic raised-panel profile and decorative half-moon glass. "
    }
]


We're ready to get back to our case study and see how adapter functions can be used in a generative program.

In [ ]:
class UnitPriceResponseFmt(pydantic.BaseModel):
    unit_price: float

class TotalPriceResponseFmt(pydantic.BaseModel):
    total_price: float

class BomEntryWithPrice(pydantic.BaseModel):
    item: str
    quantity: int | str
    notes: str
    category: Literal["lumber", "windows", "doors", "other"]
    unit_price: float | None
    total_price: float | None

def get_prices(m: mellea.MelleaSession, bom: BOM) -> list[BomEntryWithPrice]: 
    prices: list[BomEntryWithPrice] = list()
    # Go through every entry in the BOM and figure out if there's an entry in the documents.
    for i, entry in enumerate(bom.items):
        print(f"{i}/{len(bom.items) -1} ({entry.category})")
        catalog = None
        if entry.category == "windows":
            catalog = windows_doc
        elif entry.category == "doors":
            pass # uncomment to skip
            catalog = doors_doc
        elif entry.category == "lumber":
            # pass # uncomment to skip
            catalog = lumber_doc
        else:
            pass 
        
        if catalog:
            answerability_score = check_answerability(
                f"What is the price of {entry.item}?", 
                documents=[catalog], 
                context=ChatContext(),
                backend=m_hf.backend # type: ignore
            )
            if answerability_score == 'answerable':
                unit_price_response = m.instruct(
                    f"Find the `unit_price` of {entry.item} in the catalog.",
                    grounding_context={"catalog": catalog.text},
                    format=UnitPriceResponseFmt,
                )

                # Excercise: Add additional checks to ensure unit and total price are positive.
                unit_price = UnitPriceResponseFmt.model_validate_json(unit_price_response.value).unit_price
                
                total_price_respsone = m.instruct(f"Find the `total_price` given the `unit_price` and specified `quantity` for {entry.item}",
                                         grounding_context={
                                             "unit_price": str(unit_price),
                                             "quantity": str(entry.quantity)
                                         },
                                         format=TotalPriceResponseFmt)
                total_price = TotalPriceResponseFmt.model_validate_json(total_price_respsone.value).total_price
                
                prices.append(BomEntryWithPrice(
                    item=entry.item,
                    quantity=entry.quantity,
                    notes=entry.notes,
                    category=entry.category,
                    unit_price=unit_price,
                    total_price=total_price,
                ))
                continue
        
        # If we made it here then we did not succeed in adding this item's price from the catalog.
        # Exercise: use web_search tool and citation intrinsic to find prices and validate groundedness.
        # For now, we'll add back the original item with unknown prices.
        prices.append(BomEntryWithPrice(
            item=entry.item,
            quantity=entry.quantity,
            notes=entry.notes,
            category=entry.category,
            unit_price=None,
            total_price=None,
        ))
    return prices

prices = get_prices(m, bom)

0/14 (lumber)
=== 09:55:21-INFO ======
Tools for call: dict_keys([])
1/14 (lumber)
=== 09:55:37-INFO ======
Tools for call: dict_keys([])
2/14 (lumber)
=== 09:55:43-INFO ======
Tools for call: dict_keys([])


  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:55:58-INFO ======
SUCCESS


  0%|          | 0/2 [00:09<?, ?it/s]


{"unit_price": 0.0}


...

{ "unit_price": 972 }


  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:57:24-INFO ======
SUCCESS


  0%|          | 0/2 [00:02<?, ?it/s]

12/14 (doors)
13/14 (doors)


In [ ]:
# Exercise: write a requirement that enforces this constraint. Remember that quantity might be "allowance"!
print(f"Calculated price: {float(prices[-4].unit_price) * float(prices[-4].quantity)}")
print(f"LLM's price estimation: {prices[-4].total_price}")

item="3' x 4' insulated windows" quantity='6' notes='Example operable units' category='windows' unit_price=329.0 total_price=1974.0
Calculated price: 1974.0
LLM's price estimation: 1974.0


We'll stop at doors and windows to keep this tutorial brief.

_take-home exercise:_ add back lumber.

_take-home exercise:_ use Mellea's search tool to add pricing estimates for other items in the BOM.

### Step 4: Building the Final Report

In [22]:
!ollama pull gpt-oss:20b # pull gpt-oss:20b for pie chart tool calling and report construction.

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕██████████████████▏   18 B                         
pulling 776beb3adb23: 100% ▕██████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
success 


In [37]:
from mellea.backends.model_ids import OPENAI_GPT_OSS_20B
from mellea.backends.model_options import ModelOption
from mellea.stdlib.tools import local_code_interpreter
from mellea.backends.tools import MelleaTool


m = mellea.start_session(backend_name="ollama", model_id=OPENAI_GPT_OSS_20B)

report_grounding_context = {
    x.item: json.dumps({
        "total_price": x.total_price if x.total_price is not None else "unknown", 
        "category": x.category
    })
    for x in prices
}


pie_chart_result = m.instruct("Use the code interpreter tool to create a pie chart of known cost breakdowns by category. Put the pie chart in /tmp/chart.png", 
           grounding_context=report_grounding_context, tool_calls=True, # type: ignore
           model_options={ModelOption.TOOLS: [MelleaTool.from_callable(local_code_interpreter)]})

pie_chart_result.tool_calls['local_code_interpreter'].call_func() # type: ignore

report = m.instruct(
    "Write an HTML report with a top-line cost breakdown by category and a line-item material list with prices. At the top include the /tmp/chart.png image.", 
    grounding_context=report_grounding_context # type: ignore
)

open("/tmp/report.html", "w").write(report.value)

=== 09:58:58-INFO ======
Starting Mellea session: backend=ollama, model=gpt-oss:20b, context=SimpleContext


  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:58:58-INFO ======
Tools for call: dict_keys(['local_code_interpreter'])
=== 09:59:04-INFO ======
SUCCESS


  0%|          | 0/2 [00:00<?, ?it/s]

=== 09:59:29-INFO ======
SUCCESS


  0%|          | 0/2 [00:23<?, ?it/s]


3663

In [38]:
!ls /tmp/report.html

/tmp/report.html


# Module 5: Implement Your Own and Discussion

Mellea gives you the flexibility to build your own programs that meet your own specifications:
- Requirements
- Sampling Strategies
- Adapter Functions


You can also customize your programs for different goals:
- Time to First Token
- Time to First Response
- Time to First Interaction
- ...


### Resources

- **Mellea:** https://github.com/generative-computing/mellea
- **Granite Libraries:** https://huggingface.co/collections/ibm-granite/granite-libraries
- **Granite Switch:** https://github.com/generative-computing/granite-switch
- **Hugging Face (models):** https://huggingface.co/ibm-granite